# Домашнее задание: Извлечение именованных сущностей (NER) с помощью YandexGPT

**Цель:** разработать пайплайн NER для юридических документов (договоров об оказании услуг) с помощью API YandexGPT и сравнить результаты с классическими ML-моделями.

**Структура ноутбука:**
1. Подготовка окружения и данных
2. Промпты (системный и пользовательский)
3. Примеры успешного извлечения
4. Анализ граничных случаев (edge cases)
5. Сравнение с Natasha и spaCy
6. Выводы

## 1. Подготовка окружения и данных

In [17]:
import os, json, re, time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Проверка конфигурации (ключи не выводим!)
YC_FOLDER_ID = os.getenv('YC_FOLDER_ID')
YC_API_KEY   = os.getenv('YC_API_KEY')

print('YC_FOLDER_ID:', 'OK (' + YC_FOLDER_ID[:4] + '...)' if YC_FOLDER_ID else '❌ НЕ ЗАДАН')
print('YC_API_KEY:  ', 'OK (' + YC_API_KEY[:4]   + '...)' if YC_API_KEY   else '❌ НЕ ЗАДАН')

YC_FOLDER_ID: OK (b1gr...)
YC_API_KEY:   OK (AQVN...)


In [18]:
# Загрузка и диагностика документов
DOCS_DIR = Path('../data/documents')
docs = sorted(DOCS_DIR.glob('*.txt'))

print(f"{'Файл':<35} {'Симв.':>8} {'Токенов ~':>10} {'Статус':>10}")
print('-' * 70)

MAX_CHARS = 4500
for doc in docs:
    text  = doc.read_text(encoding='utf-8')
    chars = len(text)
    toks  = int(chars / 1.3)
    status = 'ok' if chars <= MAX_CHARS else 'truncate'
    print(f"{doc.name:<35} {chars:>8,} {toks:>10,} {status:>10}")

print(f'\nВсего документов: {len(docs)}')

Файл                                   Симв.  Токенов ~     Статус
----------------------------------------------------------------------
doc_01_consulting.txt                  2,243      1,725         ok
doc_02_it_support.txt                  2,042      1,570         ok
doc_03_cleaning.txt                    9,294      7,149   truncate
doc_04_marketing.txt                   1,704      1,310         ok
doc_05_accounting.txt                  2,021      1,554         ok
doc_06_nda_no_amount.txt               1,894      1,456         ok
doc_07_training_ocr_noise.txt          1,936      1,489         ok

Всего документов: 7


## 2. Промпты

### Системный промпт

Задаёт роль и поведение модели: строгость, запрет на домыслы, возврат `null` при отсутствии данных.

In [19]:
SYSTEM_PROMPT = """Ты — точный AI-помощник юриста.
Извлекай факты только из текста, без домыслов.
Если данных нет — верни null. Никогда не придумывай."""

print('=== СИСТЕМНЫЙ ПРОМПТ ===')
print(SYSTEM_PROMPT)

=== СИСТЕМНЫЙ ПРОМПТ ===
Ты — точный AI-помощник юриста.
Извлекай факты только из текста, без домыслов.
Если данных нет — верни null. Никогда не придумывай.


### Пользовательский промпт

Перечисляет целевые сущности и задаёт формат ответа через few-shot пример JSON.

In [20]:
USER_PROMPT_TEMPLATE = """Извлеки из текста договора об оказании услуг:
- parties: список сторон (name, inn, kpp)
- dates: дата подписания и сроки действия
- amounts: суммы с валютой
- obligations: только сроки обязательств (дни/месяцы/даты) если есть

Верни ТОЛЬКО валидный JSON без markdown и пояснений.
Если поля нет — верни null.

Пример:
{{
  "parties": [
    {{"name": "ООО Ромашка", "inn": "7701234567", "kpp": "770101001"}},
    {{"name": "ИП Иванов И.И.", "inn": "540312345678", "kpp": null}}
  ],
  "dates": {{"signed": "15.03.2024", "valid_from": "01.04.2024", "valid_until": "30.06.2024"}},
  "amounts": "350 000 рублей, в т.ч. НДС 20%",
  "obligations": "оплата в течение 10 рабочих дней с момента подписания акта"
}}

Текст договора:
{text}"""

print('=== ПОЛЬЗОВАТЕЛЬСКИЙ ПРОМПТ (шаблон) ===')
print(USER_PROMPT_TEMPLATE[:600], '...')

print()
print('Ключевые решения при проектировании промпта:')
print('  1. temperature=0.01  — минимальная креативность для структурных данных')
print('  2. Few-shot пример   — задаёт точную схему JSON, снижает ошибки формата')
print('  3. Явный null        — запрещает галлюцинации при отсутствии данных')
print('  4. {{double braces}} — экранирование для .format() в Python')

=== ПОЛЬЗОВАТЕЛЬСКИЙ ПРОМПТ (шаблон) ===
Извлеки из текста договора об оказании услуг:
- parties: список сторон (name, inn, kpp)
- dates: дата подписания и сроки действия
- amounts: суммы с валютой
- obligations: только сроки обязательств (дни/месяцы/даты) если есть

Верни ТОЛЬКО валидный JSON без markdown и пояснений.
Если поля нет — верни null.

Пример:
{{
  "parties": [
    {{"name": "ООО Ромашка", "inn": "7701234567", "kpp": "770101001"}},
    {{"name": "ИП Иванов И.И.", "inn": "540312345678", "kpp": null}}
  ],
  "dates": {{"signed": "15.03.2024", "valid_from": "01.04.2024", "valid_until": "30.06.2024"}},
  "amounts": "350 000 р ...

Ключевые решения при проектировании промпта:
  1. temperature=0.01  — минимальная креативность для структурных данных
  2. Few-shot пример   — задаёт точную схему JSON, снижает ошибки формата
  3. Явный null        — запрещает галлюцинации при отсутствии данных
  4. {{double braces}} — экранирование для .format() в Python


## 3. Примеры успешного извлечения

Загружаем готовые результаты из папки `results/` (предварительно запущен `ner_pipeline.py`).

In [21]:
RESULTS_DIR = Path('../results')

def show_result(doc_name: str, highlight: str = None):
    """Показывает фрагмент документа и полученный JSON рядом."""
    doc_path = DOCS_DIR / doc_name
    stem = doc_path.stem
    json_path = RESULTS_DIR / f'{stem}_ner.json'

    text = doc_path.read_text(encoding='utf-8')
    result = json.loads(json_path.read_text(encoding='utf-8')) if json_path.exists() else {'error': 'файл не найден'}

    print(f'{'='*65}')
    print(f'Документ: {doc_name}')
    print(f'{'='*65}')
    print()
    print('---  ТЕКСТ (первые 600 симв.) ---')
    print(text)
    print()
    print('--- ПОЛУЧЕННЫЙ JSON ---')
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print()

In [22]:
# Пример 1: Консультационные услуги — полный набор сущностей
show_result('doc_01_consulting.txt')

Документ: doc_01_consulting.txt

---  ТЕКСТ (первые 600 симв.) ---
ДОГОВОР № 47/К-2024
ОБ ОКАЗАНИИ КОНСУЛЬТАЦИОННЫХ УСЛУГ

г. Москва                                                           15 марта 2024 года

Общество с ограниченной ответственностью «ПравоКонсалт», именуемое в дальнейшем «Исполнитель», в лице генерального директора Смирнова Дмитрия Александровича, действующего на основании Устава, ИНН 7701234567, КПП 770101001, с одной стороны, и Общество с ограниченной ответственностью «АгроПромСтрой», именуемое в дальнейшем «Заказчик», в лице директора Петровой Ольги Николаевны, действующей на основании Устава, ИНН 7709876543, КПП 770901002, с другой стороны, заключили настоящий договор.

1. ПРЕДМЕТ ДОГОВОРА

1.1. Исполнитель обязуется оказать Заказчику консультационные услуги в области налогового законодательства и корпоративного права.
1.2. Объём, сроки и порядок оказания услуг определяются настоящим договором.

2. СТОИМОСТЬ УСЛУГ И ПОРЯДОК РАСЧЁТОВ

2.1. Стоимость услуг по насто

In [23]:
# Пример 2: IT-поддержка — ежемесячная + годовая сумма
show_result('doc_02_it_support.txt')

Документ: doc_02_it_support.txt

---  ТЕКСТ (первые 600 симв.) ---
ДОГОВОР № 112/ИТ-2024
ОБ ОКАЗАНИИ УСЛУГ ПО ТЕХНИЧЕСКОЙ ПОДДЕРЖКЕ ИНФОРМАЦИОННЫХ СИСТЕМ

г. Санкт-Петербург                                               22 января 2024 года

Акционерное общество «ИнфоТех Системс», именуемое в дальнейшем «Исполнитель», в лице генерального директора Захарова Николая Петровича, действующего на основании Устава, ИНН 7802345678, КПП 780201001, с одной стороны, и Акционерное общество «Балтийский торговый дом», именуемое в дальнейшем «Заказчик», в лице коммерческого директора Андреевой Марины Викторовны, действующей на основании Доверенности № 15 от 10.01.2024, ИНН 7804567890, КПП 780401003, с другой стороны, заключили настоящий договор.

1. ПРЕДМЕТ ДОГОВОРА

1.1. Исполнитель оказывает услуги по технической поддержке программного обеспечения «1С: Предприятие 8.3», серверной инфраструктуры и рабочих мест сотрудников Заказчика.

2. СТОИМОСТЬ УСЛУГ

2.1. Ежемесячная стоимость услуг составляет 85 

In [24]:
# Пример 3: Маркетинг — ИП без КПП (проверка null)
show_result('doc_04_marketing.txt')

Документ: doc_04_marketing.txt

---  ТЕКСТ (первые 600 симв.) ---
ДОГОВОР № 89/М-2024
ОБ ОКАЗАНИИ МАРКЕТИНГОВЫХ УСЛУГ

г. Новосибирск                                                   14 марта 2024 года

Индивидуальный предприниматель Фёдоров Игорь Викторович, именуемый в дальнейшем «Исполнитель», ОГРНИП 312543210987654, ИНН 540312345678, с одной стороны, и Общество с ограниченной ответственностью «СибирьТрейд», именуемое в дальнейшем «Заказчик», в лице генерального директора Малиновской Светланы Юрьевны, действующей на основании Устава, ИНН 5403456789, КПП 540301005, с другой стороны, заключили настоящий договор.

1. ПРЕДМЕТ ДОГОВОРА

1.1. Исполнитель оказывает услуги по продвижению Заказчика в социальных сетях ВКонтакте и Telegram, ведению таргетированной рекламы и созданию рекламных материалов.

2. СТОИМОСТЬ УСЛУГ

2.1. Стоимость услуг Исполнителя составляет 60 000 (шестьдесят тысяч) рублей в месяц, НДС не облагается.
2.2. Авансовый платёж в размере 30 000 рублей вносится до 25-го ч

## 4. Анализ граничных случаев (Edge Cases)

### 4.1 Тест на отсутствие данных — NDA без суммы

**Ожидание:** `amounts: null`, `obligations: null` — модель не должна выдумывать данные.

In [25]:
show_result('doc_06_nda_no_amount.txt')

# Проверка: amounts должен быть null
json_path = RESULTS_DIR / 'doc_06_nda_no_amount_ner.json'
if json_path.exists():
    result = json.loads(json_path.read_text(encoding='utf-8'))
    amounts = result[0].get('amounts')
    obligations = result[0].get('obligations')
    print('✅ amounts = null (нет галлюцинаций)' if amounts is None else f'❌ ГАЛЛЮЦИНАЦИЯ: amounts = {amounts}')
    print('✅ obligations != null (нет галлюцинаций)' if obligations is not None else f'⚠️  obligations = {obligations}')

Документ: doc_06_nda_no_amount.txt

---  ТЕКСТ (первые 600 симв.) ---
СОГЛАШЕНИЕ О НЕРАЗГЛАШЕНИИ КОНФИДЕНЦИАЛЬНОЙ ИНФОРМАЦИИ (NDA)

г. Ростов-на-Дону                                               10 апреля 2024 года

Общество с ограниченной ответственностью «ЮгТехнологии», именуемое в дальнейшем «Сторона 1», в лице генерального директора Кузнецова Вадима Олеговича, действующего на основании Устава, ИНН 6163345678, КПП 616301002, с одной стороны, и Общество с ограниченной ответственностью «ДонИнвест», именуемое в дальнейшем «Сторона 2», в лице директора Мещеряковой Ирины Владимировны, действующей на основании Устава, ИНН 6165678901, КПП 616501003, с другой стороны, заключили настоящее Соглашение.

1. ПРЕДМЕТ СОГЛАШЕНИЯ

1.1. Соглашение устанавливает порядок обращения Сторон с конфиденциальной информацией, раскрытой в ходе переговоров о возможном деловом сотрудничестве.
1.2. Конфиденциальной признаётся любая информация, прямо обозначенная как конфиденциальная.

2. ОБЯЗАТЕЛЬСТВА СТОРОН

2

### 4.2 Тест на зашумлённый текст — имитация плохого OCR

In [26]:
# Показываем зашумлённый фрагмент
noisy_path = DOCS_DIR / 'doc_07_training_ocr_noise.txt'
noisy_text = noisy_path.read_text(encoding='utf-8')

print('--- ЗАШУМЛЁННЫЙ ФРАГМЕНТ ---')
# Ищем строки с ИНН (там специально введены опечатки)
for line in noisy_text.splitlines():
    if 'ИНН' in line or 'инн' in line.lower() or 'КПП' in line:
        print(repr(line))  # repr покажет спецсимволы

print()
show_result('doc_07_training_ocr_noise.txt')

# Анализ: смог ли YandexGPT нормализовать зашумлённый ИНН?
json_path = RESULTS_DIR / 'doc_07_training_ocr_noise_ner.json'
if json_path.exists():
    result = json.loads(json_path.read_text(encoding='utf-8'))
    parties = result[0].get('parties', [])
    print('Извлечённые стороны:')
    for p in parties:
        print(f'  {p}')

--- ЗАШУМЛЁННЫЙ ФРАГМЕНТ ---
'Oбщество с огрaниченной ответственнoстью «УчебныЙ Центр Профессионал», имeнуемoe в дальнeйшем «Иcполнитель», в лицe дирeктора Волковa Сеpгея Анатольeвича, дeйствующeго на оснoвании Уcтава, ИНН 63О1234567, КПП 63О1О1ОО1, с однoй стороны, и Oбщество с огрaниченной ответственнoстью «ВолгaАвтoГруп», имeнуемое в дальнейшем «Заказчик», в лицe генeрального диpектора Тихонoва Алeксандра Борисовичa, дeйствующего на оснoвании Устaва, ИНН 63О9876543, КПП 63О9О1ОО2, с дpугой стoроны, зaключили нaстоящий дoговор.'
'ИНН 63О1234567, КПП 63О1О1ОО1                         ИНН 63О9876543, КПП 63О9О1ОО2'

Документ: doc_07_training_ocr_noise.txt

---  ТЕКСТ (первые 600 симв.) ---
ДOГOВОP № 5Б/ОУ-2О24
ОБ ОKАЗAНИИ ОБРАЗОВАТЕЛBНЫХ УСЛУГ

г. Самаpа                                                        О7 феврaля 2О24 гoда

Oбщество с огрaниченной ответственнoстью «УчебныЙ Центр Профессионал», имeнуемoe в дальнeйшем «Иcполнитель», в лицe дирeктора Волковa Сеpгея Анатольeвича, дeй

### 4.3 Потенциальные ошибки модели

Анализируем все результаты на наличие ошибок парсинга JSON.

In [27]:
print('=== Сканирование результатов на ошибки ===')
print()

errors_found = []
ok_count = 0

for json_file in sorted(RESULTS_DIR.glob('*_ner.json')):
    try:
        data = json.loads(json_file.read_text(encoding='utf-8'))
        # Может быть list (если один чанк вернул список)
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and 'error' in item:
                    errors_found.append((json_file.name, item))
                    break
            else:
                ok_count += 1
        elif isinstance(data, dict) and 'error' in data:
            errors_found.append((json_file.name, data))
        else:
            ok_count += 1
            print(f'  ✅ {json_file.name}')
    except json.JSONDecodeError as e:
        errors_found.append((json_file.name, {'error': str(e)}))

print()
print(f'Итого: {ok_count} успешно, {len(errors_found)} с ошибками')

if errors_found:
    print()
    print('=== Ошибки ===')
    for fname, err in errors_found:
        print(f'❌ {fname}')
        raw = err.get('raw_response', '')
        if raw:
            print(f'   Сырой ответ модели: {raw[:300]}')
        print(f'   Ошибка: {err.get("error")}')
        print()
        print('   Возможные причины и решения:')
        print('   - Модель добавила текст вокруг JSON → усилить инструкцию в промпте')
        print('   - Не закрыты скобки → добавить постобработку с json.JSONDecoder(strict=False)')
        print('   - Перепутаны ИНН сторон → добавить в промпт: "party 1 = Исполнитель, party 2 = Заказчик"')

=== Сканирование результатов на ошибки ===

  ✅ doc_03_cleaning_ner.json

Итого: 7 успешно, 0 с ошибками


## 5. Сравнение с классическими ML-моделями: Natasha и spaCy

Используем regex для извлечения ИНН/КПП и Natasha для ФИО/дат/сумм.

In [28]:
# --- Regex-подход для ИНН / КПП / р.с. ---
def extract_requisites_regex(text: str) -> dict:
    """Извлечение юридических реквизитов через регулярные выражения."""
    inn_pattern = re.compile(r'ИНН\s+(\d{10}|\d{12})(?:[,\s]+КПП\s+(\d{9}))?', re.IGNORECASE)
    rs_pattern  = re.compile(r'р/?с\s+(\d{20})', re.IGNORECASE)
    
    parties_regex = []
    for m in inn_pattern.finditer(text):
        parties_regex.append({'inn': m.group(1), 'kpp': m.group(2) or None})
    
    accounts = rs_pattern.findall(text)
    
    return {'parties_inn_kpp': parties_regex, 'bank_accounts': accounts}


# Тест на consulting-документе
sample_text = (DOCS_DIR / 'doc_01_consulting.txt').read_text(encoding='utf-8')

print('=== REGEX на doc_01_consulting.txt ===')
regex_result = extract_requisites_regex(sample_text)
print(json.dumps(regex_result, ensure_ascii=False, indent=2))
print()
print('Вывод: regex точно извлекает ИНН/КПП/р.с., но не знает контекст (Исполнитель vs Заказчик)')

=== REGEX на doc_01_consulting.txt ===
{
  "parties_inn_kpp": [
    {
      "inn": "7701234567",
      "kpp": "770101001"
    },
    {
      "inn": "7709876543",
      "kpp": "770901002"
    },
    {
      "inn": "7701234567",
      "kpp": "770101001"
    },
    {
      "inn": "7709876543",
      "kpp": "770901002"
    }
  ],
  "bank_accounts": [
    "40702810500000001234",
    "40702810300000005678"
  ]
}

Вывод: regex точно извлекает ИНН/КПП/р.с., но не знает контекст (Исполнитель vs Заказчик)


In [29]:
# --- Natasha ---
try:
    from natasha import (
        Segmenter, MorphVocab, NewsEmbedding,
        NewsMorphTagger, NewsSyntaxParser, NewsNERTagger,
        DatesExtractor, MoneyExtractor, Doc
    )
    
    segmenter    = Segmenter()
    morph_vocab  = MorphVocab()
    emb          = NewsEmbedding()
    morph_tagger = NewsMorphTagger(emb)
    ner_tagger   = NewsNERTagger(emb)
    dates_ex     = DatesExtractor(morph_vocab)
    money_ex     = MoneyExtractor(morph_vocab)

    t0  = time.perf_counter()
    doc = Doc(sample_text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.tag_ner(ner_tagger)
    for span in doc.spans:
        span.normalize(morph_vocab)
    elapsed_ms = round((time.perf_counter() - t0) * 1000)

    entities = {}
    for span in doc.spans:
        entities.setdefault(span.type, []).append(span.normal or span.text)

    dates  = [str(m.fact) for m in dates_ex(sample_text)]
    moneys = [str(m.fact) for m in money_ex(sample_text)]

    print(f'=== NATASHA на doc_01_consulting.txt ({elapsed_ms} мс) ===')
    print(json.dumps({
        'persons':       list(set(entities.get('PER', []))),
        'organizations': list(set(entities.get('ORG', []))),
        'locations':     list(set(entities.get('LOC', []))),
        'dates':         dates,
        'money':         moneys,
        'inn_kpp':       '❌ не извлекает (нет такого типа сущности)',
        'obligations':   '❌ не извлекает сроки обязательств',
    }, ensure_ascii=False, indent=2))

except ImportError:
    print('natasha не установлена: uv add natasha')

natasha не установлена: uv add natasha


In [30]:
# --- spaCy ---
try:
    import spacy
    nlp = spacy.load('ru_core_news_sm')

    t0  = time.perf_counter()
    doc = nlp(sample_text)
    elapsed_ms = round((time.perf_counter() - t0) * 1000)

    ents = {}
    for ent in doc.ents:
        ents.setdefault(ent.label_, []).append(ent.text)

    print(f'=== spaCy ru_core_news_sm на doc_01_consulting.txt ({elapsed_ms} мс) ===')
    print(json.dumps({
        'entities_by_type': {k: list(set(v)) for k, v in ents.items()},
        'inn_kpp':     '❌ не извлекает',
        'obligations': '❌ не извлекает',
    }, ensure_ascii=False, indent=2))

except (ImportError, OSError) as e:
    print(f'spaCy недоступна: {e}')
    print('Установка: uv add spacy && uv run python -m spacy download ru_core_news_sm')

=== spaCy ru_core_news_sm на doc_01_consulting.txt (80 мс) ===
{
  "entities_by_type": {
    "LOC": [
      "Москва",
      "Устава"
    ],
    "ORG": [
      "АгроПромСтрой",
      "БИК",
      "ООО «АгроПромСтрой»",
      "ОТВЕТСТВЕННОСТЬ",
      "ПАО «Сбербанк России»",
      "ПравоКонсалт",
      "РЕКВИЗИТЫ",
      "ООО «ПравоКонсалт»",
      "СТОРОН",
      "Общество с ограниченной ответственностью «",
      "АО «Альфа-Банк»",
      "ПОРЯДОК",
      "ПОДПИСИ"
    ],
    "PER": [
      "Заказчик",
      "Исполнителем",
      "Смирнов Д.А.",
      "Петровой",
      "Петрова О.Н.",
      "Смирнова",
      "Ольги Николаевны",
      "Исполнитель",
      "Дмитрия Александровича"
    ]
  },
  "inn_kpp": "❌ не извлекает",
  "obligations": "❌ не извлекает"
}


## 6. Выводы

### Готовность YandexGPT к production-задачам NER
#### ПЛЮСЫ LLM-подхода (YandexGPT):
1. Нулевой порог входа: не нужно размечать данные, обучать модель, писать правила.
   Достаточно промпта — работает «из коробки» на любых типах документов.

2. Извлекает сущности, недоступные классическим моделям:
   ИНН, КПП, р/с, сроки обязательств — типы, которых нет в Natasha и spaCy.

3. Понимает контекст: различает «Исполнитель» и «Заказчик», корректно
   привязывает реквизиты к стороне договора.

4. Устойчив к OCR-шуму: понимает «ИНН 63О1234567» (О вместо 0),
   опечатки в названиях — классические модели на таком деградируют.

5. Гибкость через промпт: новый тип сущности = правка промпта,
   а не переобучение модели.


#### МИНУСЫ LLM-подхода

1. Галлюцинации: на примерах не было обнаружено, но для более слабых моделей, и сложных документов могут быть.

2. Нестабильный JSON: на примерах не было обнаружено, но интуиция и опыт подсказывает что может быть на больших объемах и сложных документах.

3. Скорость и стоимость: 1–3 с на документ vs <200 мс у Natasha.
   При 10 000 документов/день — значимые задержки и расходы на токены.

4. Зависимость от облака: облачная модель, показала себя лучше чем llama3 
      на ноутбуке c ollama и встроенной картой с обработкой на CPU

5. Непредсказуемость: два одинаковых запроса могут дать разный результат (если не закешировано)
   (решается temperature=0.01, но не полностью).


### РЕКОМЕНДАЦИЯ ДЛЯ PRODUCTION
Оптимальный подход — ГИБРИДНЫЙ:
#### "классика"
Natasha             - ФИО, организации, даты (быстро, офлайн, без токенов) - для производства данных для валидации выхода
#### LLM
LLM(YandexGPT)      - сроки обязательств, контекст сторон, OCR-зашумлённые документы, 
      документы с структурой отличной от линейного текста (как например реквизиты, идущие в примерах не по порядку линейного чтения (ocr))

Чистый LLM-подход оправдан с участием человека для валидации.

В high-load production Natasha+regex может быть дешевле и надёжнее в автоматическом режиме для стандартных реквизитов и заранее валидированых документов с определенной структурой.
